# LeafLens training notebook

Atul Biju ASI23CA023 · Athul M B ASI23CA022 · Athul Chandran ASI23CA021

Run on a Colab GPU or in the project directory. Subset results are not a full Kaggle benchmark. This notebook contains instructions, not fabricated executed results.

In [ ]:
# On Colab, clone using your own authenticated access if the repository is private.
# !git clone https://github.com/Atul013/leaflens.git
# %cd leaflens
!pip install -r requirements.txt

## Week 1: dataset collection
Use the assigned Kaggle route. The alternative retrieves an explicitly documented upstream subset.

In [ ]:
!python scripts/download_data.py
# Alternative: !python scripts/download_subset.py --per-class 100

In [ ]:
from pathlib import Path
from ml.data import build_manifest
import collections
DATA = Path("data/PlantVillage")
manifest = build_manifest(DATA, seed=42)
print(manifest["classes"])
print(collections.Counter(row["split"] for row in manifest["samples"]))
print("Exact duplicates removed:", manifest["duplicates_removed"])

## Week 1: custom CNN baseline
Four convolutional blocks, global pooling and dropout. Check the manifest before training.

In [ ]:
!python -m ml.train --data data/PlantVillage --architecture cnn --epochs 12

## Week 2: VGG16 base training and fine-tuning
Train the head for four epochs, then unfreeze the final convolutional block at a smaller learning rate.

In [ ]:
!python -m ml.train --data data/PlantVillage --architecture vgg16 --epochs 12 --freeze-epochs 4

## Evaluation and visualization
Test images are held out from checkpoint selection. Compare split fingerprints before comparing models.

In [ ]:
import json
from IPython.display import display, Image
runs = [json.loads(Path(f"artifacts/{arch}/metrics.json").read_text()) for arch in ["cnn", "vgg16"]]
assert runs[0]["manifest_hash"] == runs[1]["manifest_hash"]
for run in runs:
    arch = run["architecture"]
    print(arch, "accuracy", run["accuracy"], "macro F1", run["macro_f1"], "ECE", run["calibration"]["ece"])
    display(Image(filename=f"artifacts/{arch}/training-curves.png"))
    display(Image(filename=f"artifacts/{arch}/confusion-matrix.png"))

In [ ]:
!python scripts/publish_results.py

## Interpretation
Controlled backgrounds do not establish field performance. Exact duplicates are removed, but physical-leaf grouping is not verified. Confidence is uncalibrated. Grad-CAM is a sensitivity map, not lesion segmentation. See docs/REPORT.md and docs/VIVA.md. Completed local measured outputs are saved separately in docs/results/.